In [1]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, v):
        if self.parent[v] != v:
            self.parent[v] = self.find(self.parent[v])
        return self.parent[v]

    def union(self, a, b):
        a = self.find(a)
        b = self.find(b)

        if a == b:
            return False

        self.parent[b] = a
        return True


def kruskal(n, edges, red_first):
    
    if red_first:
        edges = sorted(edges, key=lambda e: e[2] != "R")
    else:
        edges = sorted(edges, key=lambda e: e[2] != "B")

    dsu = DSU(n)

    tree = []
    red_count = 0

    for u, v, color in edges:

        if dsu.union(u, v):

            tree.append((u, v, color))

            if color == "R":
                red_count += 1

    if len(tree) != n - 1:
        return None, None

    return tree, red_count


def find_tree_with_k_red(n, edges, k):

    tree_min, min_red = kruskal(n, edges, red_first=False)

    _, max_red = kruskal(n, edges, red_first=True)

    if not (min_red <= k <= max_red):
        return None

    tree = tree_min[:]
    current_red = min_red

    while current_red < k:

        changed = False

        for edge in edges:

            if edge in tree or edge[2] != "R":
                continue

            u, v, _ = edge

            parent = [-1] * n
            graph = [[] for _ in range(n)]

            for a, b, c in tree:
                graph[a].append((b, c))
                graph[b].append((a, c))

            stack = [u]
            used = [False] * n
            used[u] = True

            while stack:
                x = stack.pop()

                for to, color in graph[x]:
                    if not used[to]:
                        used[to] = True
                        parent[to] = (x, color)
                        stack.append(to)

            cur = v

            while parent[cur] != -1:

                prev, color = parent[cur]

                if color == "B":

                    for rem in tree:
                        if (
                            (rem[0] == prev and rem[1] == cur) or
                            (rem[0] == cur and rem[1] == prev)
                        ) and rem[2] == "B":

                            tree.remove(rem)
                            tree.append(edge)

                            current_red += 1
                            changed = True
                            break

                if changed:
                    break

                cur = prev

            if changed:
                break

        if not changed:
            return None

    return tree


n = 5

edges = [
    (0, 1, "R"),
    (0, 2, "B"),
    (1, 2, "R"),
    (1, 3, "B"),
    (2, 3, "R"),
    (3, 4, "B"),
    (2, 4, "R")
]

k = 2

result = find_tree_with_k_red(n, edges, k)

if result is None:
    print("Решение не существует")
else:
    print("Остовное дерево:\n")

    red_count = 0

    for u, v, color in result:
        print(u, v, color)

        if color == "R":
            red_count += 1

    print("\nКрасных рёбер:", red_count)

Остовное дерево:

1 3 B
3 4 B
0 1 R
1 2 R

Красных рёбер: 2
